In [18]:
import pandas as pd
import numpy as np
import json
import duckdb
from datetime import datetime, timedelta
import pycountry
import phonenumbers

In [19]:
# Define file paths
excel_file = 'Fawlty_Luxury_Collection_Data.xlsx'
db_file = 'fawlty_luxury_collection_data.db'    

In [20]:
# Load the Excel file and check available sheets
xl = pd.ExcelFile(excel_file)
print('Sheets available:', xl.sheet_names)

# Load each sheet into a DataFrame
df_guests = xl.parse('Guests')
df_hotels = xl.parse('Hotels')
df_tiers = xl.parse('Tiers')
df_benefits = xl.parse('Benefits')
df_stays = xl.parse('Stays')
df_reviews = xl.parse('Reviews updated')
df_guests_tiers = xl.parse('Guests_Tiers')
df_stays_benefits = xl.parse('Stays_Benefits')
df_tiers_benefits = xl.parse('Tiers_Benefits')


print('Guests:', df_guests.shape)
print('Hotels:', df_hotels.shape)
print('Tiers:', df_tiers.shape)
print('Benefits:', df_benefits.shape)
print('Stays:', df_stays.shape)
print('Reviews:', df_reviews.shape)
print('Guests_Tiers:', df_guests_tiers.shape)
print('Stays_Benefits:', df_stays_benefits.shape)
print('Tiers_Benefits:', df_tiers_benefits.shape)

Sheets available: ['Guests', 'Hotels', 'Stays', 'Reviews updated', 'Guests_Tiers', 'Tiers', 'Benefits', 'Stays_Benefits', 'Tiers_Benefits']
Guests: (500, 8)
Hotels: (10, 8)
Tiers: (4, 3)
Benefits: (5, 3)
Stays: (4795, 11)
Reviews: (1841, 5)
Guests_Tiers: (500, 4)
Stays_Benefits: (8728, 4)
Tiers_Benefits: (11, 2)


In [21]:
# Prep data
df_guests['join_date'] = pd.to_datetime(df_guests['join_date']).dt.date
df_hotels['post_code'] = df_hotels['post_code'].astype(str)
df_stays['check_in_date'] = pd.to_datetime(df_stays['check_in_date']).dt.date
df_stays['comp_night'] = df_stays['comp_night'].astype(bool)
df_reviews = df_reviews[['Review_ID', 'Stay_ID', 'csat_score', 'review_text']].copy()

In [22]:
# Creating allowed value list for countries, nationalities, and country codes
countries = sorted([country.name for country in pycountry.countries])
dial_codes = sorted(set(
    phonenumbers.country_code_for_region(country.alpha_2)
    for country in pycountry.countries
    if phonenumbers.country_code_for_region(country.alpha_2) > 0
))

print(f'Countries  : {len(countries)}')
print(f'Dial codes : {len(dial_codes)}')

country_check = ",\n".join(f"'{c.replace(chr(39), chr(39)+chr(39))}'" for c in countries)
dialcode_check = ", ".join(str(c) for c in dial_codes)

print('Sample countries  :', countries[:5])
print('Sample dial codes :', dial_codes[:5])

Countries  : 249
Dial codes : 204
Sample countries  : ['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra']
Sample dial codes : [1, 7, 20, 27, 30]


In [23]:
# Setting up database connection
con = duckdb.connect(db_file)

# Drop tables in reverse dependency order (safe re-run)
drop_order = [
    'Stays_Benefits', 'Tiers_Benefits', 'Reviews',
    'Guests_Tiers', 'Stays', 'Guests', 'Hotels', 'Benefits', 'Tiers'
]
for table in drop_order:
    con.execute(f'DROP TABLE IF EXISTS {table}')
print('Existing tables dropped (if any).')

Existing tables dropped (if any).


In [24]:
# Create tables
con.execute(f"""
CREATE TABLE IF NOT EXISTS Guests (
    Guest_ID     VARCHAR(7) PRIMARY KEY,
    first_name   VARCHAR NOT NULL,
    last_name    VARCHAR NOT NULL,
    country_code INTEGER NOT NULL CHECK (country_code IN ({dialcode_check})),
    phone_number VARCHAR(25),
    email        VARCHAR NOT NULL UNIQUE,
    nationality  VARCHAR NOT NULL CHECK (nationality IN ({country_check})),
    join_date    DATE
)""")

con.execute(f"""
CREATE TABLE IF NOT EXISTS Hotels (
    Hotel_ID                 VARCHAR(7) PRIMARY KEY,
    hotel_name               VARCHAR(50) NOT NULL UNIQUE,
    address                  VARCHAR NOT NULL,
    post_code                VARCHAR(10) NOT NULL,
    country                  VARCHAR NOT NULL CHECK (country IN ({country_check})),
    total_rooms              INTEGER NOT NULL CHECK (total_rooms >= 0),
    country_code             INTEGER NOT NULL CHECK (country_code IN ({dialcode_check})),
    reception_contact_number VARCHAR(25) NOT NULL
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Tiers (
    Tier_ID         VARCHAR(7) PRIMARY KEY,
    tier_name       VARCHAR(15) NOT NULL UNIQUE,
    nights_required INTEGER NOT NULL UNIQUE CHECK (nights_required >= 0)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Benefits (
    Benefit_ID       VARCHAR(7) PRIMARY KEY,
    benefit_name     VARCHAR(50) NOT NULL UNIQUE,
    latest_unit_cost REAL NOT NULL CHECK (latest_unit_cost >= 0)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Stays (
    Stay_ID          VARCHAR(15) PRIMARY KEY,
    Guest_ID         VARCHAR(7) NOT NULL REFERENCES Guests(Guest_ID),
    Hotel_ID         VARCHAR(7) NOT NULL REFERENCES Hotels(Hotel_ID),
    check_in_date    DATE NOT NULL,
    number_of_nights INTEGER NOT NULL CHECK (number_of_nights >= 0),
    room_rate        REAL NOT NULL CHECK (room_rate >= 0),
    f_and_b_spend    REAL DEFAULT 0 CHECK (f_and_b_spend >= 0),
    spa_spend        REAL DEFAULT 0 CHECK (spa_spend >= 0),
    points_earned    INTEGER DEFAULT 0 CHECK (points_earned >= 0),
    points_redeemed  INTEGER DEFAULT 0 CHECK (points_redeemed >= 0),
    comp_night       BOOLEAN DEFAULT FALSE
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Reviews (
    Review_ID            VARCHAR(15) PRIMARY KEY,
    Stay_ID              VARCHAR(15) NOT NULL UNIQUE REFERENCES Stays(Stay_ID),
    csat_score           INTEGER NOT NULL CHECK (csat_score BETWEEN 1 AND 10),
    review_text          VARCHAR
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Guests_Tiers (
    Guest_ID            VARCHAR(7) NOT NULL REFERENCES Guests(Guest_ID),
    Tier_ID             VARCHAR(7) NOT NULL REFERENCES Tiers(Tier_ID),
    total_nights_stayed INTEGER DEFAULT 0 CHECK (total_nights_stayed >= 0),
    points_balance      INTEGER DEFAULT 0 CHECK (points_balance >= 0),
    PRIMARY KEY (Guest_ID, Tier_ID)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Tiers_Benefits (
    Tier_ID     VARCHAR(7) NOT NULL REFERENCES Tiers(Tier_ID),
    Benefit_ID  VARCHAR(7) NOT NULL REFERENCES Benefits(Benefit_ID),
    PRIMARY KEY (Tier_ID, Benefit_ID)
)""")

con.execute("""
CREATE TABLE IF NOT EXISTS Stays_Benefits (
    Stay_ID               VARCHAR(15) NOT NULL REFERENCES Stays(Stay_ID),
    Benefit_ID            VARCHAR(7) REFERENCES Benefits(Benefit_ID),
    quantity_used_benefit INTEGER DEFAULT 0 CHECK (quantity_used_benefit >= 0),
    historical_unit_cost  REAL NOT NULL CHECK (historical_unit_cost >= 0),
    PRIMARY KEY (Stay_ID, Benefit_ID)
)""")

print('All tables created successfully.')

All tables created successfully.


In [25]:
# Insert in dependency order (parents before children)
con.register('df_guests', df_guests)
con.execute('INSERT INTO Guests SELECT * FROM df_guests')
print(f'Guests inserted : {con.execute("SELECT COUNT(*) FROM Guests").fetchone()[0]} rows')

con.register('df_hotels', df_hotels)
con.execute('INSERT INTO Hotels SELECT * FROM df_hotels')
print(f'Hotels inserted : {con.execute("SELECT COUNT(*) FROM Hotels").fetchone()[0]} rows')

con.register('df_tiers', df_tiers)
con.execute('INSERT INTO Tiers SELECT * FROM df_tiers')
print(f'Tiers inserted : {con.execute("SELECT COUNT(*) FROM Tiers").fetchone()[0]} rows')

con.register('df_benefits', df_benefits)
con.execute('INSERT INTO Benefits SELECT * FROM df_benefits')
print(f'Benefits inserted : {con.execute("SELECT COUNT(*) FROM Benefits").fetchone()[0]} rows')

con.register('df_stays', df_stays)
con.execute('INSERT INTO Stays SELECT * FROM df_stays')
print(f'Stays inserted : {con.execute("SELECT COUNT(*) FROM Stays").fetchone()[0]} rows')

con.register('df_reviews', df_reviews)
con.execute('INSERT INTO Reviews SELECT * FROM df_reviews')
print(f'Reviews inserted : {con.execute("SELECT COUNT(*) FROM Reviews").fetchone()[0]} rows')

con.register('df_guests_tiers', df_guests_tiers)
con.execute('INSERT INTO Guests_Tiers SELECT * FROM df_guests_tiers')
print(f'Guests_Tiers : {con.execute("SELECT COUNT(*) FROM Guests_Tiers").fetchone()[0]} rows')

con.register('df_tiers_benefits', df_tiers_benefits)
con.execute('INSERT INTO Tiers_Benefits SELECT * FROM df_tiers_benefits')
print(f'Tiers_Benefits : {con.execute("SELECT COUNT(*) FROM Tiers_Benefits").fetchone()[0]} rows')

con.register('df_stays_benefits', df_stays_benefits)
con.execute('INSERT INTO Stays_Benefits SELECT * FROM df_stays_benefits')
print(f'Stays_Benefits : {con.execute("SELECT COUNT(*) FROM Stays_Benefits").fetchone()[0]} rows')

print('All data inserted successfully.')

Guests inserted : 500 rows
Hotels inserted : 10 rows
Tiers inserted : 4 rows
Benefits inserted : 5 rows
Stays inserted : 4795 rows
Reviews inserted : 1841 rows
Guests_Tiers : 500 rows
Tiers_Benefits : 11 rows
Stays_Benefits : 8728 rows
All data inserted successfully.


In [26]:
# List all tables in the DB
con.close()
con = duckdb.connect(db_file)
tables = con.execute("SHOW TABLES").df()
print('Tables in database:')
display(tables)

Tables in database:


,name
0,Benefits
1,Guests
2,Guests_Tiers
3,Hotels
4,Reviews
5,Stays
6,Stays_Benefits
7,Tiers
8,Tiers_Benefits


In [27]:
# Insight 1: The Benefit Cannibalization & Upsell Test
# Query 1A: The F&B Cannibalization Check
con.execute("""
WITH Stay_Breakfast_Flag AS (
    SELECT
        s.Stay_ID,
        s.number_of_nights,
        s.f_and_b_spend,
        CASE WHEN sb.Benefit_ID IS NOT NULL THEN 'Has Free Breakfast'
             ELSE 'No Free Breakfast' END AS Breakfast_Status,
        CASE WHEN s.f_and_b_spend > 0 THEN 1.0 ELSE 0.0 END AS Has_FB_Spend
    FROM Stays s
    LEFT JOIN Stays_Benefits sb
        ON s.Stay_ID = sb.Stay_ID
        AND sb.Benefit_ID = (SELECT Benefit_ID FROM Benefits WHERE benefit_name = 'Free breakfast')
    WHERE s.comp_night = 0
)
SELECT
    Breakfast_Status,
    COUNT(Stay_ID)                              AS Total_Stays,
    ROUND(AVG(f_and_b_spend / number_of_nights), 2) AS Avg_Daily_FB_Spend,
    ROUND(AVG(Has_FB_Spend) * 100, 2)          AS FB_Penetration_Pct
FROM Stay_Breakfast_Flag
GROUP BY Breakfast_Status
""").df()

,Breakfast_Status,Total_Stays,Avg_Daily_FB_Spend,FB_Penetration_Pct
0,Has Free Breakfast,1067,59.07,50.42
1,No Free Breakfast,3384,126.06,76.74


In [28]:
# Query 1B: The Spa Upsell Check
con.execute("""
WITH Stay_Spa_Flag AS (
    SELECT
        s.Stay_ID,
        s.number_of_nights,
        s.spa_spend,
        CASE WHEN sb.Benefit_ID IS NOT NULL THEN 'Has Free Spa Access'
             ELSE 'No Free Spa Access' END AS Spa_Status,
        CASE WHEN s.spa_spend > 0 THEN 1.0 ELSE 0.0 END AS Has_Spa_Spend
    FROM Stays s
    LEFT JOIN Stays_Benefits sb
        ON s.Stay_ID = sb.Stay_ID
        AND sb.Benefit_ID = (SELECT Benefit_ID FROM Benefits WHERE benefit_name = 'Free Spa access')
    WHERE s.comp_night = 0
)
SELECT
    Spa_Status,
    COUNT(Stay_ID)                               AS Total_Stays,
    ROUND(AVG(spa_spend / number_of_nights), 2)  AS Avg_Daily_Spa_Spend,
    ROUND(AVG(Has_Spa_Spend) * 100, 2)           AS Spa_Penetration_Pct
FROM Stay_Spa_Flag
GROUP BY Spa_Status
""").df()

,Spa_Status,Total_Stays,Avg_Daily_Spa_Spend,Spa_Penetration_Pct
0,No Free Spa Access,4158,105.11,56.45
1,Has Free Spa Access,293,103.77,77.82


In [29]:
# Insight 2: Customer Lifetime Value (CLV) & The Annual Restructure
# Query 2: The Temporal Velocity Check (Accumulation vs. Active Year)
con.execute("""
WITH Guest_Lifetimes AS (
    SELECT
        s.Guest_ID,
        SUM(s.number_of_nights) AS Lifetime_Nights,
        SUM(CASE WHEN EXTRACT(YEAR FROM s.check_in_date) = 2025
                 THEN s.number_of_nights ELSE 0 END) AS Year_2025_Nights,
        SUM(COALESCE(sb.quantity_used_benefit * sb.historical_unit_cost, 0)) AS Lifetime_Benefit_Cost
    FROM Stays s
    LEFT JOIN Stays_Benefits sb ON s.Stay_ID = sb.Stay_ID
    GROUP BY s.Guest_ID
)
SELECT
    t.tier_name                        AS Current_Loyalty_Tier,
    COUNT(g.Guest_ID)                  AS Total_Guests_In_Tier,
    ROUND(AVG(g.Lifetime_Nights), 1)   AS Avg_Lifetime_Nights,
    ROUND(AVG(g.Year_2025_Nights), 1)  AS Avg_Active_Nights_2025
FROM Guest_Lifetimes g
JOIN Guests_Tiers gt ON g.Guest_ID = gt.Guest_ID
JOIN Tiers t          ON gt.Tier_ID = t.Tier_ID
WHERE t.tier_name <> 'Basic'
GROUP BY t.tier_name, gt.Tier_ID
ORDER BY gt.Tier_ID ASC
""").df()

,Current_Loyalty_Tier,Total_Guests_In_Tier,Avg_Lifetime_Nights,Avg_Active_Nights_2025
0,Silver,175,23.7,23.3
1,Gold,100,65.6,64.1
2,Diamond,75,172.9,110.5


In [30]:
# Insight 3: The CSAT Paradox & Unstructured Sentiment Mining
# Query 3A: Phase 1 — The Quantitative Trigger
con.execute("""
SELECT
    b.benefit_name           AS Loyalty_Benefit,
    COUNT(r.Review_ID)       AS Total_Reviews,
    ROUND(AVG(r.csat_score), 2) AS Average_CSAT_Score
FROM Benefits b
JOIN Stays_Benefits sb ON b.Benefit_ID = sb.Benefit_ID
JOIN Stays s           ON sb.Stay_ID   = s.Stay_ID
JOIN Reviews r         ON s.Stay_ID    = r.Stay_ID
GROUP BY b.benefit_name
ORDER BY Average_CSAT_Score DESC
""").df()

,Loyalty_Benefit,Total_Reviews,Average_CSAT_Score
0,Free Wifi,1841,6.81
1,Welcome drinks,988,6.61
2,Free breakfast,538,6.47
3,Free Spa access,164,6.15
4,Complimentary room upgrade,67,5.40


In [31]:
# Query 3B: Phase 2 — The Qualitative Deep Dive
con.execute("""
WITH Upgrade_Reviews AS (
    -- Step 1: Isolate reviews that specifically mention upgrades
    SELECT
        Review_ID,
        csat_score,
        LOWER(review_text) AS text
    FROM Reviews
    WHERE LOWER(review_text) LIKE '%upgrade%'
),
Complaint_Categorization AS (
    SELECT
        Review_ID,
        csat_score,
        CASE
            WHEN csat_score > 6
                THEN 'Positive Upgrade Experience'
            WHEN text LIKE '%available%' 
            OR text LIKE '%denied%' 
            OR text LIKE '%refused%'
                THEN 'Theme 1: Denied/No Availability'
            WHEN text LIKE '%view%' 
            OR text LIKE '%floor%' 
            OR text LIKE '%same%'
                THEN 'Theme 2: Marginal/Fake Upgrade'
            ELSE 'Theme 3: General Upgrade Complaint'
        END AS Upgrade_Sentiment_Theme
    FROM Upgrade_Reviews
),
Negative_Total AS (
    SELECT COUNT(*) AS n
    FROM Complaint_Categorization
    WHERE csat_score <= 6
)
-- Step 3: Aggregate for the business report
SELECT
    Upgrade_Sentiment_Theme,
    COUNT(Review_ID)                        AS Total_Reviews,
    ROUND(AVG(csat_score), 2)               AS Average_CSAT,
    ROUND(
        (COUNT(Review_ID) * 100.0) /
        NULLIF((SELECT n FROM Negative_Total), 0),
    1)                                      AS Pct_Of_Negative_Mentions
FROM Complaint_Categorization
WHERE Upgrade_Sentiment_Theme != 'Positive Upgrade Experience'
GROUP BY Upgrade_Sentiment_Theme
ORDER BY Total_Reviews DESC
""").df()

,Upgrade_Sentiment_Theme,Total_Reviews,Average_CSAT,Pct_Of_Negative_Mentions
0,Theme 3: General Upgrade Complaint,460,3.70,88.3
1,Theme 1: Denied/No Availability,61,3.97,11.7


In [32]:
# Query 3B: Phase 2 — The Qualitative Deep Dive (Refined with AI-Identified Wildcards)
con.execute("""
WITH Upgrade_Reviews AS (
    -- Step 1: Isolate reviews that specifically mention upgrades
    SELECT
        Review_ID,
        csat_score,
        LOWER(review_text) AS text
    FROM Reviews
    WHERE LOWER(review_text) LIKE '%upgrade%'
),
Complaint_Categorization AS (
    -- Step 2: Categorize into 4 meaningful themes based on root cause
    SELECT
        Review_ID,
        csat_score,
        text,
        CASE
            WHEN csat_score > 6
            THEN 'Positive Upgrade Experience'
            WHEN text LIKE '%availab%'
                 OR text LIKE '%subject to%'
                 OR text LIKE '%cited%'
                 OR text LIKE '%fully%'
            THEN 'Theme 1: No Availability'
            WHEN text LIKE '%status%'
                 OR text LIKE '%loyal%'
                 OR text LIKE '%member%'
                 OR text LIKE '%tier%'
                 OR text LIKE '%benefit%'
                 OR (text LIKE '%ignored%' AND text LIKE '%expectation%')
            THEN 'Theme 2: Loyalty/Benefits Not Recognized'
            WHEN (text LIKE '%received%'
                  OR text LIKE '%did offer%'
                  OR text LIKE '%provided%'
                  OR text LIKE '%offered%')
                 AND (text LIKE '%but%'
                      OR text LIKE '%however%'
                      OR text LIKE '%not ready%'
                      OR text LIKE '%not worth%')
            THEN 'Theme 3: Upgrade Unsatisfactory'
            ELSE 'Theme 4: Miscellaneous'
        END AS Upgrade_Complaint_Theme
    FROM Upgrade_Reviews
),
Total_Complaints AS (
    SELECT COUNT(*) AS n
    FROM Complaint_Categorization
    WHERE csat_score <= 6 
)
-- Step 3: Aggregate for the business report
SELECT
    Upgrade_Complaint_Theme,
    COUNT(Review_ID) AS Total_Reviews,
    ROUND(AVG(csat_score), 2) AS Average_CSAT,
    ROUND(
        (COUNT(Review_ID) * 100.0) / 
        NULLIF((SELECT n FROM Total_Complaints), 0),
    1) AS Pct_Of_All_Complaints
FROM Complaint_Categorization
WHERE Upgrade_Complaint_Theme != 'Positive Upgrade Experience'
GROUP BY Upgrade_Complaint_Theme
ORDER BY Total_Reviews DESC
""").df()

,Upgrade_Complaint_Theme,Total_Reviews,Average_CSAT,Pct_Of_All_Complaints
0,Theme 2: Loyalty/Benefits Not Recognized,391,3.64,75.0
1,Theme 1: No Availability,124,4.00,23.8
2,Theme 4: Miscellaneous,4,4.00,0.8
3,Theme 3: Upgrade Unsatisfactory,2,4.00,0.4


In [33]:
# Close the connection
con.close()
print(f'Database saved and connection closed: {db_file}')

Database saved and connection closed: fawlty_luxury_collection_data.db
